# 03. LLM 노출 추출과 정답 라벨 비교 (파일럿 P2)

사람이 작성·커밋한 정답 라벨(`labels_P2.csv`)과 같은 (발행사, 테마) 쌍을 LLM으로 판정하고, 프로토콜 7.1의 P2 기준으로 비교한다.

**설계 (실행 전 고정)**
- 판정 기준: `labeling_guide.md`의 2~7절 원문을 그대로 프롬프트에 넣는다. 사람과 같은 기준을 쓰게 하기 위함이다.
- 입력: 02 노트북이 만든 **테마별 키워드 적중 문단 전체**. 사람이 읽기 보조로 쓴 것과 같은 자료다.
  사람은 적중 문단 밖의 범위도 읽을 수 있었다는 차이가 있으며, 이 차이는 한계로 기록한다.
- 출력: JSON 스키마로 고정(label, 방향, 인용, 문단 ID, 이유). 호출마다 원본 응답과 프롬프트 해시를 저장한다.

**잠금 장치**
- 정답 라벨이 미작성·검증 실패·미커밋·커밋 후 수정 상태이면 LLM을 호출하지 않는다.
- 첫 실행이 **공식 결과**다. 결과를 보고 다시 돌려 좋은 결과를 고르는 일을 막기 위해, 이미 실행 기록이 있으면 기본적으로 다시 실행하지 않는다.

In [ ]:
# ── 설정 ─────────────────────────────────────────────
USE_FIXTURES = False            # True: 가짜 LLM·가짜 라벨로 흐름만 점검 (실제 결과 아님)
MODEL = "claude-sonnet-5-5"     # 실행 후에는 바꾸지 않는다 (run_meta.json 에 기록됨)
MAX_TOKENS = 2000
NEW_RUN = False                 # True: 공식 실행이 이미 있어도 새로 실행 (사유를 experiment_log.md 에 기록할 것)

In [ ]:
import sys, json, os, hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests")]
import kic13f as k
import filings as fl
import llm_extract as L

GUIDE_PATH = ROOT / "labeling_guide.md"
guide_text = GUIDE_PATH.read_text(encoding="utf-8")
rules = L.guide_rules(guide_text)

if USE_FIXTURES:
    from fixtures import FakeClient, FakeLLM, demo_labels
    sec = FakeClient()
    sample = pd.read_csv(ROOT / "data" / "processed_fixture" / "S1" / "sample.csv").sort_values("rank")
    docs = {}
    for r in sample.head(5).itertuples(index=False):
        url = fl.primary_document_url(sec, r.cik, r.latest_10k_before_cutoff)
        docs[int(r.cik)] = fl.assign_items(fl.html_to_paragraphs(sec.get_bytes(url)))
    labels = demo_labels(sample, docs)
    LABEL_DIR = ROOT / "data" / "labeling_fixture"
    LABEL_PATH = None
    print("⚠ 모의 모드: 가짜 라벨과 가짜 LLM. 결과는 의미 없음.")
else:
    sample = pd.read_csv(ROOT / "data" / "processed" / "S1" / "sample.csv").sort_values("rank")
    LABEL_DIR = ROOT / "data" / "labeling"
    LABEL_PATH = LABEL_DIR / "labels_P2.csv"
    labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
    docs = {}
    for r in sample[sample["cik"].isin(labels["cik"].astype(int))].itertuples(index=False):
        p = LABEL_DIR / "docs" / f"{r.cik}_{r.latest_10k_before_cutoff}.csv"
        docs[int(r.cik)] = pd.read_csv(p, encoding="utf-8-sig", keep_default_na=False)   # 02 노트북과 같은 문단 ID

labels["rank"] = labels["rank"].astype(int); labels["cik"] = labels["cik"].astype(int)
print(f"정답 라벨 {len(labels)}건:", labels["label"].value_counts().to_dict())

## 1. 실행 전 잠금 확인

In [ ]:
validation = fl.validate_labels(labels, docs) if not USE_FIXTURES else pd.DataFrame({"ok": [True] * len(labels)})
git_info = None if USE_FIXTURES else L.git_label_commit(ROOT, LABEL_PATH)
problems = L.gate_check(labels, validation, git_info)
print("정답 라벨 커밋:", git_info)
if problems:
    raise RuntimeError("LLM 실행 불가:\n- " + "\n- ".join(problems))
print("(모의 모드: 커밋 잠금 검사 생략)" if USE_FIXTURES else "✓ 잠금 해제: 정답 라벨이 검증·커밋된 상태")

## 2. 비용 추정

In [ ]:
prompts = [L.build_prompt(rules, r.company, r.theme, fl.keyword_hits(docs[r.cik], r.theme))
           for r in labels.itertuples(index=False)]
in_tok = sum(len(s) + len(u) for s, u in prompts) / 4          # 대략 4글자 = 1토큰 (한국어는 더 많을 수 있음)
out_tok = len(prompts) * MAX_TOKENS
print(f"호출 {len(prompts)}회, 입력 약 {in_tok:,.0f} 토큰, 출력 최대 {out_tok:,} 토큰")
print(f"예상 비용 상한 (Sonnet 5.5 기준 $2/$10 per MTok): 약 ${in_tok/1e6*2 + out_tok/1e6*10:.2f}")

## 3. LLM 실행

API 키는 환경변수 `ANTHROPIC_API_KEY`에서 읽는다. 없으면 입력창이 뜬다. **키를 노트북이나 파일에 적지 않는다.**

In [ ]:
RUNS_DIR = LABEL_DIR / "llm_runs"
existing = sorted(p for p in RUNS_DIR.glob("*") if (p / "predictions.csv").exists()) if RUNS_DIR.exists() else []

if existing and not NEW_RUN:
    print("공식 실행이 이미 있다 → 새로 실행하지 않음:", existing[0].name)
else:
    if USE_FIXTURES:
        llm = FakeLLM()
    else:
        import anthropic, getpass
        if not os.environ.get("ANTHROPIC_API_KEY"):
            os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
        llm = anthropic.Anthropic()
    run_dir = RUNS_DIR / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir.mkdir(parents=True)
    meta = {"model": MODEL, "max_tokens": MAX_TOKENS, "started_at_utc": datetime.now(timezone.utc).isoformat(),
            "labels_commit": (git_info or {}).get("commit"), "labels_committed_at": (git_info or {}).get("committed_at"),
            "guide_sha256": hashlib.sha256(guide_text.encode()).hexdigest(),
            "protocol": "experiment_protocol.md v1.0", "guide": "labeling_guide.md v1.1",
            "input_design": "테마별 키워드 적중 문단 전체 (filings.keyword_hits)",
            "official": not existing, "fixture": USE_FIXTURES}
    (run_dir / "run_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    preds = L.run_extraction(llm, MODEL, labels, docs, rules, run_dir, MAX_TOKENS)
    print("실행 완료:", run_dir.relative_to(ROOT), "| 오류", int((preds["llm_label"] == "error").sum()), "건")
    existing = sorted(p for p in RUNS_DIR.glob("*") if (p / "predictions.csv").exists())

## 4. 정답 라벨과 비교 (프로토콜 7.1 P2)

**공식 결과는 첫 번째 실행**이다. 통과 조건: 인용 원문 일치율 100%, 전체 판정 일치 80% 이상, confirmed가 아닌 사례(4건 이상)의 판정 일치 75% 이상.

In [ ]:
official = existing[0]
preds = pd.read_csv(official / "predictions.csv", encoding="utf-8-sig", keep_default_na=False)
res = L.evaluate(labels, preds, docs)
m = res["metrics"]
print("공식 실행:", official.name)
for key, label in [("quote_verbatim_rate", "인용 원문 일치율"), ("overall_agreement", "전체 판정 일치"),
                   ("non_confirmed_agreement", "confirmed가 아닌 사례 일치")]:
    print(f"  {label}: {m[key]:.1%}")
print(f"  (인용은 원문에 있으나 문단 ID 가 다른 경우: {m['quote_wrong_pid']}건, LLM 응답 오류: {m['llm_errors']}건)")
print("\nP2 판정:", "통과" if m["p2_pass"] else "미통과")
display(res["confusion"])
display(res["by_theme"])

In [ ]:
cols = ["rank", "company", "theme", "label", "llm_label", "match", "llm_quote_status",
        "exposure_direction", "llm_direction", "llm_evidence_location", "llm_rationale"]
display(res["detail"][cols])
res["detail"].to_csv(official / "evaluation_detail.csv", index=False, encoding="utf-8-sig")
(official / "evaluation.json").write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", (official / "evaluation.json").relative_to(ROOT))

In [ ]:
# experiment_log.md 에 붙여넣을 줄
print(f"| {datetime.now():%Y-%m-%d %H:%M} | P2 LLM 비교 | 모델 {MODEL}, 실행 {official.name}, 정답 라벨 커밋 "
      f"{(git_info or {}).get('commit', '')[:10]} | {'통과' if m['p2_pass'] else '미통과'} — 인용 {m['quote_verbatim_rate']:.0%}, "
      f"전체 {m['overall_agreement']:.0%}, 비confirmed {m['non_confirmed_agreement']:.0%} (n={m['non_confirmed_n']}) | |")

## 5. 해석 시 유의점

- 정답 라벨 15건 남짓의 소규모 파일럿이다. 일치율은 파이프라인의 초기 품질 점검이며, 제품 성능으로 일반화하지 않는다.
- 사람은 적중 문단 밖도 읽을 수 있었고 LLM은 적중 문단만 받았다. 불일치가 이 입력 차이에서 왔는지 `evaluation_detail.csv`로 사례별로 확인한다.
- LLM은 대상 기업에 대한 사전지식을 가지고 있다. 인용 의무로 일부 통제하지만 완전히 차단되지는 않는다(프로토콜 9절).
- 미통과 시: 프로토콜 7.1에 따라 테마 정의 또는 추출 방식을 수정하고 v1.1로 기록한다. **공식 결과는 지우지 않는다.**
- `data/labeling/llm_runs/`의 실행 기록 전체를 커밋한다.